# How are period and amplitude related?

In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from scipy import stats
from stats_toolkit import pearson_r

rr = pd.read_csv("../data/raw/gaia_dr3_rrlyrae_pilot_raw.csv")
rr["period"] = rr["pf"].fillna(rr["p1_o"])
rr["period_type"] = np.where(rr["pf"].notna(), "fundamental", "overtone")
rr["log_period"] = np.log10(rr["period"])
print(rr.shape)

(2000, 13)


In [2]:
fund = rr[rr["period_type"] == "fundamental"].dropna(subset=["log_period", "peak_to_peak_g"])

mine = pearson_r(fund["log_period"], fund["peak_to_peak_g"])
theirs = stats.pearsonr(fund["log_period"], fund["peak_to_peak_g"])[0]

print("n =", len(fund))
print("mine  :", mine)
print("scipy :", theirs)
print("difference:", abs(mine - theirs))

n = 1262
mine  : -0.3820991923788068
scipy : -0.38209919237880685
difference: 5.551115123125783e-17


In [3]:
from stats_toolkit import correlation_summary

res = correlation_summary(fund["log_period"], fund["peak_to_peak_g"])
for k, v in res.items():
    print(k, ":", v)

# check the interval against scipy
sp = stats.pearsonr(fund["log_period"], fund["peak_to_peak_g"])
print("scipy 95% CI:", sp.confidence_interval())

n : 1262
r : -0.3820991923788068
r2 : 0.1459997928165364
se : 0.02403965875881792
pe : 0.016214749832822686
ci95 : (np.float64(-0.42825196419373496), np.float64(-0.33395821716533786))
scipy 95% CI: ConfidenceInterval(low=np.float64(-0.42825113532491954), high=np.float64(-0.3339591189849056))


In [4]:
import pandas as pd

rr2 = rr.dropna(subset=["log_period", "peak_to_peak_g"])
groups = {
    "pooled": rr2,
    "fundamental": rr2[rr2["period_type"] == "fundamental"],
    "overtone": rr2[rr2["period_type"] == "overtone"],
}

rows = []
for name, d in groups.items():
    s = correlation_summary(d["log_period"], d["peak_to_peak_g"])
    rows.append({"group": name, "n": s["n"], "r": s["r"], "r2": s["r2"],
                 "ci_low": s["ci95"][0], "ci_high": s["ci95"][1]})

pd.DataFrame(rows).round(3)

,group,n,r,r2,ci_low,ci_high
0,pooled,2000,0.403,0.163,0.366,0.439
1,fundamental,1262,-0.382,0.146,-0.428,-0.334
2,overtone,738,0.218,0.047,0.148,0.285
